### Simulación de Eventos de Seguridad Bancaria
Implementación con bloqueo de 60 segundos tras 3 intentos fallidos, visualización en tiempo real del flujo de actividades y cálculo dinámico del tiempo restante de baneo.

In [ ]:
import time
import random
from datetime import datetime

class BancoStream:
    """
    Clase encargada de simular la generación continua de eventos de un banco.
    """
    def __init__(self):
        # Definimos una lista pequeña de usuarios para forzar colisiones y probar los bloqueos
        self.usuarios = [f"user_{i}" for i in range(1, 4)]
        # Tipos de acciones posibles en la plataforma bancaria
        self.acciones = ["login", "logout", "transferencia", "pago_servicio"]

    def generar_evento(self):
        """
        Genera un único evento aleatorio simulando la actividad de un cliente.
        """
        accion = random.choice(self.acciones)
        # Aumentamos la probabilidad de 'login' para facilitar la demostración de bloqueos
        if random.random() < 0.40:
            accion = "login"

        # El estado de la acción puede ser exitoso o fallido (50% de probabilidad)
        estado = "exitoso" if random.random() > 0.50 else "fallido"

        return {
            "usuario": random.choice(self.usuarios),     # Selecciona un usuario al azar
            "accion": accion,                            # Acción realizada
            "timestamp": datetime.now(),                 # Fecha y hora legible para visualización
            "event_time": time.time(),                   # Timestamp Unix (segundos) para cálculos lógicos
            "estado": estado                             # Resultado de la acción
        }

    def stream(self, n_eventos=40, delay=0.5):
        """
        Generador que produce eventos uno por uno con un intervalo de tiempo (delay).
        """
        for _ in range(n_eventos):
            yield self.generar_evento()
            time.sleep(delay) # Pausa para simular el paso del tiempo real en el flujo

class ProcesadorEventosBancarios:
    """
    Procesador que analiza en tiempo real los eventos recibidos y aplica las reglas de negocio
    y seguridad (detección de intrusos, baneos por intentos fallidos y cálculo de tiempo restante).
    """
    MAX_INTENTOS = 3       # Número máximo de intentos permitidos antes de aplicar baneo
    TIEMPO_BLOQUEO = 60    # Duración del baneo en segundos

    def __init__(self):
        self.intentos_fallidos = {}  # Diccionario para almacenar: usuario -> cantidad de logins fallidos
        self.bloqueados_hasta = {}   # Diccionario para almacenar: usuario -> timestamp exacto en que finaliza su baneo
        self.event_window = []       # Ventana de almacenamiento deslizante para los últimos eventos procesados

    def procesar_evento(self, evento):
        """
        Procesa un evento individual, valida el estado de baneo del usuario y actualiza los contadores.
        """
        usuario = evento["usuario"]
        accion = evento["accion"]
        ahora = evento["event_time"]
        # Formateamos la hora en formato legible HH:MM:SS.mmm
        hora_legible = evento["timestamp"].strftime("%H:%M:%S.%f")[:-3]
        estado = evento["estado"]

        # --- GESTIÓN DE VENTANA DESLIZANTE ---
        # Almacenamos el evento en la ventana
        self.event_window.append(evento)
        # Si la ventana supera los 10 elementos, descartamos el más antiguo (deslizamiento de ventana)
        if len(self.event_window) > 10:
            self.event_window.pop(0)

        # Imprimir en pantalla el detalle de cada evento recibido
        print(f"[EVENTO] [{hora_legible}] - Usuario: {usuario:8} | Acción: {accion:15} | Estado: {estado}")

        # --- CONTROL DE BANEO ACTIVO ---
        if usuario in self.bloqueados_hasta:
            tiempo_restante = self.bloqueados_hasta[usuario] - ahora

            if tiempo_restante > 0:
                # Si el usuario sigue baneado e intenta iniciar sesión de nuevo
                if accion == "login":
                    print(f"  -> [ACCESO DENEGADO] {usuario} intentó ingresar, pero está BANEADO. Tiempo restante: {tiempo_restante:.1f} segundos.")
                # Interrumpimos el flujo para este evento ya que el usuario está bloqueado
                return
            else:
                # Si el tiempo de baneo ya pasó, levantamos el castigo de forma automática
                print(f"  -> [DESBLOQUEADO] El baneo de {usuario} ha expirado. Restableciendo intentos...")
                del self.bloqueados_hasta[usuario]
                self.intentos_fallidos[usuario] = 0

        # --- CONTROL DE INTENTOS DE LOGIN ---
        if accion == "login":
            if estado == "fallido":
                # Incrementamos el contador de fallos para el usuario
                self.intentos_fallidos[usuario] = self.intentos_fallidos.get(usuario, 0) + 1
                intentos = self.intentos_fallidos[usuario]
                print(f"  -> [ALERTA] Intento de login FALLIDO ({intentos}/{self.MAX_INTENTOS}) para {usuario}")

                # Evaluamos si alcanzamos el límite para banear
                if intentos >= self.MAX_INTENTOS:
                    # Calculamos el momento exacto en el futuro en que se levantará el bloqueo
                    self.bloqueados_hasta[usuario] = ahora + self.TIEMPO_BLOQUEO
                    print(f"  -> [BANEADO 🚫] {usuario} ha superado el límite. Suspendido por {self.TIEMPO_BLOQUEO} segundos.")
            else:
                # Si el login es exitoso, reiniciamos su contador de fallas a cero
                self.intentos_fallidos[usuario] = 0
                print(f"  -> [LOGIN EXITOSO] {usuario} ha ingresado correctamente.")

In [ ]:
# --- BLOQUE DE EJECUCIÓN DE LA SIMULACIÓN ---

# Instanciamos el generador de eventos (Simulador de streaming)
stream_banco = BancoStream()

# Instanciamos el procesador que contiene toda la lógica de baneo y ventanas de tiempo
procesador = ProcesadorEventosBancarios()

print("=== INICIANDO MONITOREO EN TIEMPO REAL ===\n")

# Iniciamos la lectura del flujo constante de eventos (en este caso definimos un límite de 35 eventos)
# Usamos un delay de 0.8 segundos entre eventos para emular la llegada progresiva de transacciones
for evento in stream_banco.stream(n_eventos=35, delay=0.8):
    procesador.procesar_evento(evento)

print("\n=== SIMULACIÓN FINALIZADA ===")

=== INICIANDO MONITOREO EN TIEMPO REAL ===

[EVENTO] [08:43:09.813] - Usuario: user_3   | Acción: transferencia   | Estado: exitoso
[EVENTO] [08:43:10.613] - Usuario: user_1   | Acción: login           | Estado: exitoso
  -> [LOGIN EXITOSO] user_1 ha ingresado correctamente.
[EVENTO] [08:43:11.413] - Usuario: user_1   | Acción: pago_servicio   | Estado: fallido
[EVENTO] [08:43:12.214] - Usuario: user_1   | Acción: transferencia   | Estado: fallido
[EVENTO] [08:43:13.014] - Usuario: user_3   | Acción: pago_servicio   | Estado: fallido
[EVENTO] [08:43:13.814] - Usuario: user_2   | Acción: login           | Estado: fallido
  -> [ALERTA] Intento de login FALLIDO (1/3) para user_2
[EVENTO] [08:43:14.615] - Usuario: user_1   | Acción: login           | Estado: fallido
  -> [ALERTA] Intento de login FALLIDO (1/3) para user_1
[EVENTO] [08:43:15.415] - Usuario: user_1   | Acción: login           | Estado: exitoso
  -> [LOGIN EXITOSO] user_1 ha ingresado correctamente.
[EVENTO] [08:43:16.216] - 